In [11]:
import os
from databricks.connect import DatabricksSession

# Initialize Spark locally when needed
if "spark" not in globals():
    spark = DatabricksSession.builder.serverless().getOrCreate()

#NotebookParameters
if "dbutils" in globals():
    #Execute in Databricks
    dbutils.widgets.dropdown(
        "environment", 
        "dev", 
        ["dev", "staging", "prod"]
    )

    dbutils.widgets.text("organization", "databricks")

    environment = dbutils.widgets.get("environment")
    organization = dbutils.widgets.get("organization").strip()
else:
    #Execute in VSCode with Databricks Connect
    environment = os.getenv("LAKEHOUSE_ENV", "dev")
    organization = os.getenv("GITHUB_ORGANIZATION", "databricks").strip()

catalogs = {
    "dev": "opensource_dev",
    "staging": "opensource_staging",
    "prod": "opensource_prod"
}

catalog = catalogs[environment]
schema_name = "github_analytics"

bronze_table = f"{catalog}.{schema_name}.bronze_github_repositories"
silver_staging_table = f"{catalog}.{schema_name}.silver_staging_github_repositories"

print(f"Source Table: {bronze_table}")
print(f"Target Table: {silver_staging_table}")

Source Table: opensource_dev.github_analytics.bronze_github_repositories
Target Table: opensource_dev.github_analytics.silver_staging_github_repositories


In [12]:
from pyspark.sql import functions as F


#Read Bronze data for the selected organization
df_bronze = (
    spark.read.table(bronze_table)
    .filter(F.col("organization") == organization)
)

#Retrieve the two most recent snapshots for the selected organization
snapshots = (
    df_bronze
    .select("snapshot_id", "ingestion_ts")
    .distinct()
    .orderBy(
        F.col("ingestion_ts").desc(),
        F.col("snapshot_id").desc()
    )
    .limit(2)
    .collect()
)

if not snapshots:
    raise ValueError(f"No snapshots found for organization '{organization}' in Bronze table '{bronze_table}'.")

#Current Snapshot
current_snapshot_id = snapshots[0]["snapshot_id"]

df_current = df_bronze.filter(F.col("snapshot_id") == current_snapshot_id)

#Previous Snapshot
if len(snapshots) > 1:
    previous_snapshot_id = snapshots[1]["snapshot_id"]
else:
    previous_snapshot_id = None
#Read previous snapshot data if it exists
if previous_snapshot_id is not None:
    df_previous = df_bronze.filter(F.col("snapshot_id") == previous_snapshot_id)
else:
    df_previous = None

print(f"Current snapshot: {current_snapshot_id}")
print(f"Previous snapshot: {previous_snapshot_id}")
print(f"Current records: {df_current.count()}")
if df_previous is not None:
    print(f"Previous records: {df_previous.count()}")
else:
    print("No previous snapshot available.")

Current snapshot: 6a7b4914-f985-4953-a2e8-d79a3ba182d0
Previous snapshot: 943612ea-bc21-4671-b03e-b437e6f2482b
Current records: 253
Previous records: 253


In [ ]:


from pathlib import Path
import sys

# Find the project root by searching for the shared Change Detection module
# Starting from the current working directory, check all parent directories
project_root = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "src" / "change_detection.py").is_file()
    ),
    None
)

# Raise an error if the shared module cannot be found
if project_root is None:
    raise FileNotFoundError(
        "Cannot locate src/change_detection.py"
    )

# Add the project root to Python's module search path if not already present
# This allows importing shared modules from the src directory
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import the shared Change Detection function
# The same function is used by the Databricks pipeline and PySpark unit tests
from src.change_detection import detect_repository_changes

# Compare current and previous snapshots to classify repository changes
# Returns NEW, UPDATED or UNCHANGED records using null-safe comparisons
df_changes = detect_repository_changes(
    df_current,
    df_previous
)

df_changes.groupBy("change_type").count().show()


+-----------+-----+
|change_type|count|
+-----------+-----+
|  UNCHANGED|  253|
+-----------+-----+



In [16]:
#Select only new or update repositories
df_changed = df_changes.filter(F.col("change_type").isin("NEW", "UPDATED"))

#Display the change distribution
df_changed.groupBy("change_type").count().show()

print(f"Repositories to process: {df_changed.count()}")

+-----------+-----+
|change_type|count|
+-----------+-----+
+-----------+-----+

Repositories to process: 0


In [17]:
#Define the Change Detection target table
changes_table = f"{catalog}.{schema_name}.silver_changes_github_repositories"

#Save new and updated repositories
(
    df_changed
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(changes_table)

)

print(f"Changes saved to table: {changes_table}")
print(f"Records written: {df_changed.count()}")

Changes saved to table: opensource_dev.github_analytics.silver_changes_github_repositories
Records written: 0


In [18]:
df_check = spark.read.table(changes_table)

df_check.groupBy("change_type").count().show()

+-----------+-----+
|change_type|count|
+-----------+-----+
+-----------+-----+

